## Introduction
This milestone focuses on the practical application of fine-tuning a large language model using synthetic data for a specific task: generating structured JSON output that adheres to a predefined schema. The goal is to guide the model in producing test cases that evaluate insurance policy configurations for rating-related issues. This exercise demonstrates the use of domain-specific examples to tailor a model’s behavior, aligning its outputs with internal business needs while working within the constraints of not disclosing proprietary information.

The workflow involved the following key steps:
1. **Schema Definition:** A target output schema was defined and saved as `output_schema.json`, representing the desired format for the model's outputs (see: Appendix B).
2. **Synthetic Data Generation:** A `.jsonl` dataset was manually created with input/output pairs to simulate realistic prompts and their expected JSON-formatted responses (see Appendix A).
3. **Fine-Tuning `(covered by this notebook)`:** A pre-trained base model was fine-tuned using the synthetic dataset to learn how to produce structured outputs aligned with the defined schema.
4. **Evaluation:** The outputs of the fine-tuned model were compared against those of the base model to assess improvements in format adherence and semantic accuracy.

Sections **7 (Model Performance Evaluation)** and **8 (Reflection and Future Considerations)** provide in-depth analysis and discussion on the effectiveness of the fine-tuning effort. These sections also explore potential root causes for observed limitations and offer insights into how incorporating proprietary data and documentation could improve model fidelity.

## 1. Setup

Import the necessary libraries. Nothing to see here—just Python packages.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import os
import json
import pprint
import tiktoken

### Step 1.1 Setting Up the Variables

Here I use dotenv to load the environment file and make my API key available.  I then create a client using OpenAI.  This client will be used to perform the remaining steps of the assignment.  

In addition to creating the client, I also create some global variables that will be used throughout the notebook.  I chose to use the `gpt-4o-mini-2024-07-18` model as the base model for fine-tuning due to its relatively low cost.  I looked up in the OpenAI Pricing page that the cost for this model to be fine-tuned is about `$3.00` per `1,000,000 tokens`.  And, finally, I set the base path to the file here so that it doesn't need to be typed into many different places throughout the code.

In [ ]:
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

model_name = os.getenv("BASE_MODEL_NAME", "gpt-4o-mini-2024-07-18")
if not model_name:
    st.error("BASE_MODEL_NAME not found in environment; please check your .env file.")
    st.stop()

model_suffix = os.getenv("MODEL_SUFFIX", "darkwave-synthetic-data-v1")
if not model_suffix:
    st.error("MODEL_SUFFIX not found in environment; please check your .env file.")
    st.stop()

# Global Variables for the project
model = model_name
MILLION_TOKENS = 1_000_000
# change for your model
price_per_1M_tokens = 3.00

# Training File Path
file_path = '../json/policy_scenarios.jsonl'

## 2. Estimate Cost of Fine Tuning

We will attempt to count tokens to estimate the training cost.  I am using a JSONL file that contains examples I created based on domain knowledge from my employer.  The json schema used does not exactly represent a working rating structure, the intent was to have many policy characteristics that would be used across insurance companies.  

### 2.1 Defining Functions

Here, I define two functions to help count tokens and estimate cost.  The first function will taken in the file that contains the training data and the name of the model to be used in fine-tuning; if no model is passed the `gpt-4o-mini-2024-07-18` model will be used as the default.  This function will loop through the training data and split the messages into role and content values that will be tokenized as part of training.  I also added `assistant\n` to the string to encode because I found an online reference indicating this might help.  However, I still do not believe that these estimates are accurate and I need to do additional research as stated above.  Primarily, I need to account for: 10% validation holdout which should not be charged as training tokens and any special characters that GPT may include in tokenization (e.g., <|start|>, etc).

In [5]:
def count_tokens_in_jsonl(file_path, model="gpt-4o-mini-2024-07-18"):
    """
    Count the number of tokens in a JSONL file using tiktoken.
    Args:
        file_path (str): Path to the JSONL file.
        model (str): The model to use for tokenization (default is gpt-4o-mini-2024-07-18).
    Returns:
        int: The total number of tokens in the file.
    """
    # Initialize the tokenizer for the specified model
    enc = tiktoken.encoding_for_model(model)
    
    total_tokens = 0
    
    # Open the JSONL file
    with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            # Assuming 'messages' field contains the chat conversation
            conversation = data.get("messages", [])
            for message in conversation:
                role = message.get("role", "")
                content = message.get("content", "")

                # Lightweight concatenation: role and content (NO <|start|> <|end|>)
                combined = f"{role}\n{content}\nassistant\n"
                
                # Tokenize the content and count the tokens
                total_tokens += len(enc.encode(combined))
    
    return total_tokens

def estimate_fine_tuning_cost(tokens, price_per_1M_tokens=3.00):
    """
    Estimate the cost of fine-tuning based on the number of tokens.
    Args:
        tokens (int): Number of tokens.
        price_per_1M_tokens (float): Price per 1,000,000 tokens in dollars.
    Returns:
        float: Estimated fine-tuning cost in dollars.
    """
    return (tokens / MILLION_TOKENS) * price_per_1M_tokens


### 2.2 Perform the Count and Estimate

Here we call the count and estimate functions and output the results to the user.  We will see that this implementation gets us pretty close to the expected output.

In [6]:
tokens = count_tokens_in_jsonl(file_path, model)
cost = estimate_fine_tuning_cost(tokens, price_per_1M_tokens)

print(f"Total tokens: {tokens}")
print(f"Estimated cost of fine-tuning: ${cost:.4f}")

Total tokens: 27042
Estimated cost of fine-tuning: $0.0811


### 2.3 Token Counting Accuracy

As we will see later on, this estimation did not exactly match the ChatGPT token count used to train the model.  In a professional setting, I would spend more time looking over the [Token Counting Cookbook](https://cookbook.openai.com/examples/how_to_count_tokens_with_tiktoken) to see if I could glean additional insights into how to get a closer estimate.

## 3. Upload the File

Here we upload the file to ChatGPT using the Files API.  We will need the file id to pass to the fine-tuning job so that it can use the data in training.

***Note: Once the file was successfully loaded, I commented out and hard coded in the file ID to reduce clutter in the OpenAI Playground***

In [ ]:
file_response = client.files.create(
    file=open(file_path, "rb"),
    purpose="fine-tune"
)

file_id = file_response.id

# Comment above code and use below if model is 
#    already trained and you know the file ID
#    saves on upload time
# file_id='your file id here'

In [ ]:
# swap if you need the uploaded file ID
# print(f"Uploaded file ID: {file_id}")
print(f"Uploaded file ID: <redacted>")

Uploaded file ID: <redacted>


## 4. Create the Fine-Tuning Job

Here we are going to create the fining tuning job.  It will use the model defined at the top of the notebook in the training.  We also pass the `"n_epochs": 1` hyperparameter to ensure the model only runs once.  I decided to do this to keep the cost low.  By default Fine-Tuning will run 3 epochs which essentially triples the number of tokens used since all tokens are used in each epoch.  I also provided a `suffix: "dsc670-week09-pricing-ft"` job parameter which provides some control over the model name generated by fine-tuning.  That said, I'm not sure it really was all that beneficial aside from exploring different job/hyperparameters that could be used.

***Note: After the fine-tuning job was created, I commented out the function and hard-coded the fine_tune_job.id to ensure there were not additional costs when re-running the notebook.***

In [ ]:
fine_tune_job = client.fine_tuning.jobs.create(
    training_file=file_id,
    model=model,
    hyperparameters={
        "n_epochs": 1
    },
    suffix=model_suffix,
)

fine_tune_job_id = fine_tune_job.id

# Comment above and use below if model tune job 
#    created and you know the job ID saves on 
#    cost by not creating multiple jobs
# fine_tune_job_id = 'your job id here'


In [ ]:
# swap if you need the fine-tune job ID
# print(f"Fine-tuning job ID: {fine_tune_job_id}")
print(f"Fine-tuning job ID: <redacted>")

Fine-tuning job ID: <redacted>


## 5. Monitoring Job Completion

### 5.1 List Events

In order to monitor job completion, there are a couple of different APIs that can be used.  In this section, I am using the `list_events` API and monitoring the job messages as each event processes.  This allowed me to follow along as the job ran to see how close/far from completion it might be.  It also notified me when the job was complete.

In [44]:
job_response = client.fine_tuning.jobs.list_events(fine_tune_job_id)
for event in job_response.data[:2] + job_response.data[3:]: # Skip the event which contains model name
    print(f"Message: {event.message}")

Message: The job has successfully completed
Message: Usage policy evaluations completed, model is now enabled for sampling
Message: Evaluating model against our usage policies
Message: New fine-tuned model created
Message: Step 40/40: training loss=0.28
Message: Step 39/40: training loss=0.25
Message: Step 38/40: training loss=0.29
Message: Step 37/40: training loss=0.27
Message: Step 36/40: training loss=0.30
Message: Step 35/40: training loss=0.27
Message: Step 34/40: training loss=0.27
Message: Step 33/40: training loss=0.29
Message: Step 32/40: training loss=0.20
Message: Step 31/40: training loss=0.27
Message: Step 30/40: training loss=0.29
Message: Step 29/40: training loss=0.29
Message: Step 28/40: training loss=0.29
Message: Step 27/40: training loss=0.30
Message: Step 26/40: training loss=0.29


### 5.2 Job Status

In addition to monitoring the job as it runs using `list_events` you can also just monitor the job status using the `retrieve` API and inspecting the response status.  In this case, I'm storing the response from the `retrieve` API in a variable called `status` and then printing out `status.status`.  As the job runs, this status will change state.  A few of the observed states were: "Job Status: Validating Data", "Job Status: Running", and "Job Status: Succeeded" (as shown below).  If there is no need to monitor the individual events in the job, you could just jump to the status and wait for a "succeeded" status (or failure status) to monitor job completion.

In [20]:
status = client.fine_tuning.jobs.retrieve(fine_tune_job_id)

print(f"Job status: {status.status}")

Job status: succeeded


## 6. Fine-Tuning Report

The following code will create a fine-tuning report that can be viewed by the user.  This detail can be obtained from the `retrieve` APIs response.  For this report, I included the Model ID, Status, Training Tokens Used, Timestamp Job Started, Timestamp Job Finished, Hyperparameters used in fine-tuning, Estimated Fine Tuning Cost, and cost per 1 million tokens I was able to find on the OpenAI pricing page.

In [22]:
# Retrieve the job details
job_info = client.fine_tuning.jobs.retrieve(fine_tune_job_id)

# Extract key details
ft_model_name = job_info.fine_tuned_model
ft_training_tokens = job_info.trained_tokens
ft_hyperparameters = job_info.hyperparameters
ft_status = job_info.status
ft_created_at = job_info.created_at
ft_finished_at = job_info.finished_at

# Your known total cost from billing (replace this if needed)

total_cost_usd = (ft_training_tokens / MILLION_TOKENS) * price_per_1M_tokens

# Print Training Report
print("\nFine-Tuning Training Report\n")
print(f"Fine-tuned Model ID: <redacted>")
# print(f"Fine-tuned Model ID: {ft_model_name}")
print(f"Job Status: {ft_status}")
print(f"Training Tokens Used: {ft_training_tokens:,}")
print(f"Training Started: {ft_created_at}")
print(f"Training Finished: {ft_finished_at}")
print(f"Hyperparameters:")
print(f"  - Batch Size: {ft_hyperparameters.batch_size}")
print(f"  - Learning Rate Multiplier: {ft_hyperparameters.learning_rate_multiplier}")
print(f"  - Number of Epochs: {ft_hyperparameters.n_epochs}")
print(f"Estimated Fine-Tuning Cost: ${total_cost_usd:.4f}")
print(f"Cost per 1 Million Tokens: ${price_per_1M_tokens:.2f}")



Fine-Tuning Training Report

Fine-tuned Model ID: <redacted>
Job Status: succeeded
Training Tokens Used: 27,082
Training Started: 1763602946
Training Finished: 1763603275
Hyperparameters:
  - Batch Size: 1
  - Learning Rate Multiplier: 1.8
  - Number of Epochs: 1
Estimated Fine-Tuning Cost: $0.0812
Cost per 1 Million Tokens: $3.00


## 7. Using the Model

Here, I use the completions API to call the fine-tuned model.  The objective of this section is to evaluate the performance of the fine-tuned model when applied to synthetic prompts simulating insurance policy descriptions. The key focus is on whether the model can produce structured outputs in the expected JSON format that resemble plausible test cases.

### 7.1 Fine-Tuning Example 1


In [23]:
completion = client.chat.completions.create(
    model=ft_model_name,
    messages=[
        {"role": "system", "content": "You are a JSON generator. Respond only with valid JSON following the provided schema."},
        {"role": "user", "content": "Create a policy in CA with poor billing history."}
    ],
    max_tokens=2000
)

print(completion.choices[0].message.content)

{"policies": [{"policy_id": "CA051013", "policy_period_start_date": "2025-02-26", "policy_period_expiration_date": "2026-02-26", "policy_original_inception_date": "2021-02-27", "policy_latest_cancellation_date": null, "policy_status": "cancelled", "prior_insurance_carrier": "SafeInsure", "prior_insurance_status": "active", "prior_insurance_tenure_in_months": 69, "prior_insurance_expiration_date": "2023-09-02", "policy_state_code": "CA", "prior_policy_term_premium_amt": 434.1, "policy_line_of_business": "PersonalAuto", "policy_line_of_business_product_version_id": "v1.0", "policy_line_of_business_sku_id": "sku-001", "drivers": [{"driver_id": "CA05101301", "license_status": "Valid", "license_issue_date": "1971-02-27", "driver_dob": "1995-02-27", "driver_annual_mileage": 11945.64, "driver_financial_responsibility_indicator": true, "driver_driving_experience_start_date": "1985-02-27", "accidents": [], "convictions": []}], "vehicles": [{"vehicle_id": "CA05101301", "vehicle_vin": "VIN0000000

### 7.2 Fine-Tuning Example 2

In [24]:
completion = client.chat.completions.create(
    model=ft_model_name,
    messages=[
        {"role": "system", "content": "You are a JSON generator. Respond only with valid JSON following the provided schema."},
        {"role": "user", "content": "Create a policy in IL with one driver and one vehicle used for work/school."}
    ],
    max_tokens=2000
)

print(completion.choices[0].message.content)

{"policies": [{"policy_id": "POL005", "policy_period_start_date": "2023-06-14", "policy_period_expiration_date": "2024-06-14", "policy_original_inception_date": "2021-06-15", "policy_latest_cancellation_date": null, "policy_status": "active", "prior_insurance_carrier": "Insure All", "prior_insurance_status": "active", "prior_insurance_tenure_in_yrs": 32.76, "prior_insurance_expiration_date": "2024-05-20", "policy_state_code": "IL", "prior_policy_term_premium_amt": 296.63, "policy_line_of_business": "PersonalAuto", "policy_line_of_business_product_version_id": "v1.0", "drivers": [{"driver_id": "DR00501", "driver_birth_month_start_date": "1996-07-01", "license_status": "Valid", "license_issue_date": "1996-07-01", "driver_annual_mileage": 6086.01, "driver_financial_responsibility_indicator": true, "driver_driving_experience_months": 309.5, "driver_gender": "Female", "marital_status": "Single", "credit_score_range": "Good", "accidents": [], "convictions": []}], "vehicles": [{"vehicle_id": 

### 7.3 Base Model Examples

Here, I pass the two examples above that were passed to the Fine-Tuned model to the base model (`gpt-4o-mini`) that was tuned for the assignment.  As you will see, the base model returns the actual emoji icon while the fine-tuned model returns the tuned format.

#### 7.3.1 FT Example 1

In [25]:
completion = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a JSON generator. Respond only with valid JSON following the provided schema."},
        {"role": "user", "content": "Create a policy in CA with poor billing history."}
    ],
    max_tokens=2000
)

print(completion.choices[0].message.content)

{
  "policy": {
    "state": "CA",
    "billingHistory": "poor",
    "premium": {
      "amount": 750,
      "currency": "USD",
      "paymentFrequency": "monthly"
    },
    "coverage": {
      "type": "auto",
      "limits": {
        "liability": {
          "bodilyInjury": {
            "perPerson": 30000,
            "perAccident": 60000
          },
          "propertyDamage": 15000
        },
        "comprehensive": {
          "deductible": 500
        },
        "collision": {
          "deductible": 500
        }
      }
    },
    "deductions": [
      {
        "type": "latePayment",
        "amount": 100
      },
      {
        "type": "lapsedCoverage",
        "amount": 150
      }
    ],
    "status": "active",
    "startDate": "2023-10-01",
    "endDate": "2024-10-01"
  }
}


#### 7.3.2 FT Example 2

In [26]:
completion = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a JSON generator. Respond only with valid JSON following the provided schema."},
        {"role": "user", "content": "Create a policy in IL with one driver and one vehicle used for work/school."}
    ],
    max_tokens=2000
)

print(completion.choices[0].message.content)

{
  "policy": {
    "state": "IL",
    "drivers": [
      {
        "name": "John Doe",
        "age": 30,
        "license_number": "D123456789",
        "driving_experience_years": 10,
        "email": "johndoe@example.com",
        "phone_number": "555-1234"
      }
    ],
    "vehicles": [
      {
        "make": "Toyota",
        "model": "Camry",
        "year": 2020,
        "vin": "1HGBH41JXMN109186",
        "usage": "work/school"
      }
    ],
    "coverage": {
      "liability": {
        "bodily_injury": {
          "per_person": 100000,
          "per_accident": 300000
        },
        "property_damage": 50000
      },
      "collision": {
        "deductible": 500
      },
      "comprehensive": {
        "deductible": 500
      },
      "uninsured_motorist": {
        "bodily_injury": {
          "per_person": 100000,
          "per_accident": 300000
        }
      }
    },
    "policy_details": {
      "policy_number": "IL123456789",
      "start_date": "2023-10-01"

### 7.4 Fine-Tuned Model vs. Base Model
Compared to the base model, which frequently produced unstructured or irrelevant text, the fine-tuned model shows noticeable improvement in formatting. It consistently attempts to output JSON-like structures and uses field names that appear to be learned from the training examples. This indicates some level of understanding of the desired format and domain-specific vocabulary.

However, the outputs from the fine-tuned model still exhibit inconsistencies:
* For a given prompt, not all entities and fields appear to be fully populating (e.g., getting only one driver when asking for 2).
* Some responses include fields with unexpected values, casing or nesting.
* The data content does not always reflect valid or meaningful test scenarios, despite the correct structure.

#### Potential Root Causes
The underlying issues likely stem from the manually created .jsonl training data. These examples were synthesized to avoid disclosing proprietary details of internal rating logic. As a result:
* The volume of examples may be insufficient for the model to generalize reliably.
* Not all examples were schema-validated or semantically accurate.
* Inconsistencies in field naming and structure may have confused the model during training.

## 8. Evaluation and Reflection

**Evaluation:** The fine-tuned model reliably outputs data in JSON format, which meets the formatting objective. However, the content of the generated JSON lacks validity as test cases. While syntactically correct, the output often exhibits internal inconsistencies (e.g., repeated or incorrectly structured fields) and does not map well to real-world policy conditions. This suggests a need to revisit the quality and consistency of the training examples.

**Training Data Commentary:** The .jsonl training examples were manually crafted based on domain knowledge but were not thoroughly validated. Given the proprietary nature of the underlying rating algorithm, examples were intentionally abstracted and may not represent actual rating scenarios. As a result, the model lacked exposure to a coherent, repeatable schema or valid business logic.

**Next Steps and Potential Enhancements:** The model's performance could be substantially improved by introducing validated internal datasets. Proprietary documentation such as rate manuals, rate pages, known test cases, and REST API schema definitions could provide richer, more diverse training signals. With access to hundreds of internally curated examples, the model could more effectively learn the nuanced structures and logic embedded in real-world underwriting processes.


# Appendix
The appendix contains three representative examples from the `.jsonl` dataset used for training, along with the output schema that served as the target for model fine-tuning. The examples were manually created and not all were validated for accuracy against real policy configurations. This approach was taken to preserve the confidentiality of proprietary rating algorithms and product definitions.

## Appendix A: Output JSONL

In [27]:
max_lines = 3  # or however many you want
with open(file_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= max_lines:
            break
        data = json.loads(line)
        print(json.dumps(data, indent=2))

{
  "messages": [
    {
      "role": "system",
      "content": "You are a JSON generator. Respond only with valid JSON following the provided schema."
    },
    {
      "role": "user",
      "content": "Create a policy with two drivers: one 16-year-old and one 30-year-old driver."
    },
    {
      "role": "assistant",
      "content": "{\"policies\": [{\"policy_id\": \"POL001\", \"policy_period_start_date\": \"2025-01-01\", \"policy_period_expiration_date\": \"2026-01-01\", \"policy_original_inception_date\": \"2023-01-02\", \"policy_latest_cancellation_date\": null, \"policy_status\": \"cancelled\", \"prior_insurance_carrier\": \"SafeGuard\", \"prior_insurance_status\": \"active\", \"prior_insurance_tenure_in_months\": 117, \"prior_insurance_expiration_date\": \"2024-09-16\", \"policy_state_code\": \"TX\", \"prior_policy_term_premium_amt\": 1316.68, \"policy_line_of_business\": \"PersonalAuto\", \"policy_line_of_business_product_version_id\": \"v1.0\", \"drivers\": [{\"driver_id\

### Appendix B: Intended JSON Schema

In [30]:
print(os.getcwd())
with open("../json/schema.json", "r") as f:
    schema = json.load(f)

print(json.dumps(schema, indent=2))

c:\codebase\portfolio\Data-Science-Projects\ai-synthetic-data\milestone3
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "type": "object",
  "properties": {
    "policies": {
      "type": "array",
      "items": {
        "$ref": "#/definitions/Policy"
      }
    }
  },
  "required": [
    "policies"
  ],
  "definitions": {
    "Policy": {
      "type": "object",
      "properties": {
        "policy_id": {
          "type": "string"
        },
        "policy_period_start_date": {
          "type": "string",
          "format": "date"
        },
        "policy_period_expiration_date": {
          "type": "string",
          "format": "date"
        },
        "policy_original_inception_date": {
          "type": "string",
          "format": "date"
        },
        "policy_latest_cancellation_date": {
          "type": [
            "string",
            "null"
          ],
          "format": "date"
        },
        "policy_status": {
          "type": "string",
  